In [1]:
from pathlib import Path
import sys
from pprint import pprint

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from analysis import PosetAnalyzer
from families import boolean_lattice
from ideal import Ideal
from weighted import WeightedPoset, WeightedPosetAnalyzer

b2 = boolean_lattice(2)
b2_analyzer = PosetAnalyzer(b2)

pprint(b2_analyzer.summary())


{'height': 3,
 'lattice_layer_sizes': [1, 1, 2, 1, 1],
 'num_elements': 4,
 'num_ideals': 6,
 'num_linear_extensions': 2,
 'num_maximals': 1,
 'num_minimals': 1,
 'num_relations': 4,
 'width': 2}


## Notes
This creates a boolean lattice on a 2-element set, then runs the base analyzer on it. It eventually prints out a base summary of key poset features. For context, we are using the boolean lattices in these examples for their 'shape,' instead of their usual subset interpretation. 

We can see that this summary provides `height`, which here counts the maximum number of elements in a chain, and `width`, which is measured by the largest number of incomparable elements you can have at once. Dependency edges are counted by `num_relations`, while `lattice_layer_sizes` show how many ideals there are per rank. Valid full completions are enumerated by `num_linear_extensions`. 

More simply, in the case of task-scheduling, we know: there are 4 things we need to do, the biggest dependency chain is 3 items long, and there are 2 valid ways of completing the full task list while respecting all dependencies. The lattice layers let us see where the concurrency bottleneck manifests, in the middle at rank 2. 



In [2]:
b3 = boolean_lattice(3)
b3_analyzer = PosetAnalyzer(b3)

b3_summaries = {
    "structural_summary": b3_analyzer.summary(),
    "interval_summary": b3_analyzer.interval_summary(),
    "zeta_summary": b3_analyzer.zeta_summary(),
    "mobius_summary": b3_analyzer.mobius_summary(),
    "lattice_layer_sizes": b3_analyzer.lattice_layer_sizes(),
}

pprint(b3_summaries)


{'interval_summary': {'interval_size_histogram': {1: 8, 2: 12, 4: 6, 8: 1},
                      'max_interval_size': 8,
                      'mean_interval_size': 2.3703703703703702,
                      'min_interval_size': 1,
                      'num_cover_intervals': 12,
                      'num_intervals': 27,
                      'num_nontrivial_intervals': 19,
                      'num_trivial_intervals': 8},
 'lattice_layer_sizes': [1, 1, 3, 3, 4, 3, 3, 1, 1],
 'mobius_summary': {'is_ranked': True,
                    'mobius_abs_sum': 27,
                    'mobius_max': 1,
                    'mobius_min': -1,
                    'mobius_negative_count': 13,
                    'mobius_nonzero_count': 27,
                    'mobius_positive_count': 14,
                    'mobius_value_histogram': {-1: 13, 1: 14},
                    'mobius_value_histogram_by_rank_distance': {0: {1: 8},
                                                                1: {-1: 12},
 

## Notes
Now let's look at the other kinds of analysis we can do. This time, we'll work on the Boolean lattice of a 3-element set. $B_3$ has increased its height and width by 1 compared to $B_2$, but between its 8 elements and 12 relations, it's linear extensions explode up to 48. So this time around, we'll look into the other summaries we have included, and see what insights they bring.

In the `interval_summary` we can an idea of its structure by looking at intervals, which are subposets containing exactly the elements that fall between a pair of comparable elements in the ordering. Here we can see the `max_interval_size` matches up to the full poset in this case, while the `min_interval_size` represents the individual vertices. Edges are represented in `num_cover_intervals` while `num_intervals`, `num_trivial_intervals`, `num_nontrivial_intervals` show the total intervals, the singletons, and the non-singleton intervals. The big insight comes from the `interval_size_histogram` which shows us intervals by size. For $B_3$ we get our first hint of its cube shape, as there are 8 points, 12 edges, 6 square faces represented by intervals of size 4, all in one cube. Finally we can look at `mean_interval_size`, which is one of the ways we can heuristically understand how 'connected' our overall poset is, by giving us an idea of how hard it is to *separate* it.

The `zeta_summary` focuses on the transitive closure behind comparability. It separates stored cover relations from all strict comparable pairs, so `transitive_closure_extra_count` shows how many relations are implied by transitivity rather than directly stored in the Hasse diagram. The principal ideal and filter histograms show how much upward and downward mass each element sees in the zeta matrix.





In [3]:
def subset_size(label):
    if label == "{}":
        return 0
    return len(label.strip("{}").split(", "))


positive_weighted = WeightedPoset.from_element_function(
    b3,
    lambda element: subset_size(element) + 1,
)
mixed_weighted = positive_weighted.with_element_weights(
    lambda element: subset_size(element) - 1,
)

positive_analyzer = WeightedPosetAnalyzer(positive_weighted)
mixed_analyzer = WeightedPosetAnalyzer(mixed_weighted)

def error_message(call):
    try:
        call()
    except ValueError as exc:
        return str(exc)
    return None


weighted_showcase = {
    "same_underlying_shape": positive_weighted.poset is mixed_weighted.poset is b3,
    "positive_element_weights": positive_weighted.element_weights,
    "positive_measurements": {
        "max_chain_weight": positive_analyzer.max_chain_weight(),
        "weighted_width": positive_analyzer.weighted_width(),
        "ideal_weight_of_bottom_and_singletons": positive_analyzer.ideal_weight(
            Ideal({"{}", "{1}", "{2}", "{3}"})
        ),
        "lattice_layer_weight_summary": positive_analyzer.weighted_lattice_layer_summary(),
    },
    "mixed_element_scores": mixed_weighted.element_weights,
    "mixed_score_methods": {
        "max_chain_score": mixed_analyzer.max_chain_score(),
        "max_antichain_score": mixed_analyzer.max_antichain_score(),
        "ideal_score_of_bottom_and_singletons": mixed_analyzer.ideal_score(
            Ideal({"{}", "{1}", "{2}", "{3}"})
        ),
        "lattice_layer_score_summary": mixed_analyzer.weighted_lattice_layer_score_summary(),
    },
    "mixed_measurement_rejections": {
        "max_chain_weight": error_message(mixed_analyzer.max_chain_weight),
        "weighted_width": error_message(mixed_analyzer.weighted_width),
        "ideal_weight": error_message(
            lambda: mixed_analyzer.ideal_weight(Ideal({"{}", "{1}"}))
        ),
    },
}

pprint(weighted_showcase)


{'mixed_element_scores': {'{1, 2, 3}': 2,
                          '{1, 2}': 1,
                          '{1, 3}': 1,
                          '{1}': 0,
                          '{2, 3}': 1,
                          '{2}': 0,
                          '{3}': 0,
                          '{}': -1},
 'mixed_measurement_rejections': {'ideal_weight': 'Ideal weight requires '
                                                  'nonnegative element '
                                                  'weights.',
                                  'max_chain_weight': 'Chain weight requires '
                                                      'nonnegative element '
                                                      'weights.',
                                  'weighted_width': 'Weighted width requires '
                                                    'nonnegative element '
                                                    'weights.'},
 'mixed_score_methods': {'ideal_score_of_bott

## Notes


